# 06c Final Physics-Informed Ridge Baseline Consolidation

This notebook consolidates the final linear baseline for the fixed discrete-`n` path.

Constraints for this milestone:
- dataset unchanged (dense discrete-`n`),
- no new data generation,
- Ridge regression only,
- feature sets restricted to A/B/C from milestone 06b.

Interpretation scope is restricted to the validated parameter window of the dense discrete-`n` dataset.
This is **final baseline consolidation**, not universal final model selection.

In [ ]:
from pathlib import Path
import sys

import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.model import (
    PHYSICAL_FEATURE_SET_NAMES,
    evaluate_ridge_feature_sets,
    select_best_feature_set_by_mae,
)


In [ ]:
data_path = PROJECT_ROOT / "data" / "superellipse_discrete_n_dense_dataset.npz"
data = np.load(data_path)

n_all = data["n"].astype(float)
a_all = data["a"].astype(float)
r_all = data["aspect_ratio"].astype(float)

targets = {
    "E0": data["E0"].astype(float),
    "dE1": data["dE1"].astype(float),
    "dE3": data["dE3"].astype(float),
    "dE2": data["dE2"].astype(float),  # diagnostic-only in this milestone
}

n_classes = [1.2, 2.0, 3.0, 4.0]
main_targets = ["E0", "dE1", "dE3"]
diagnostic_targets = ["dE2"]
protocols = ["LOAO", "LOARO"]

print(f"Loaded: {data_path}")
print(f"Samples: {n_all.size}")
print(f"n classes: {np.unique(n_all)}")
print(f"Feature sets: {PHYSICAL_FEATURE_SET_NAMES}")


In [ ]:
def compact(x: float) -> str:
    return f"{x:.4e}"


def format_table(rows: list[dict[str, object]], cols: list[str]) -> str:
    if not rows:
        return "(no rows)"
    widths = {c: max(len(c), max(len(str(r[c])) for r in rows)) for c in cols}
    header = " | ".join(c.ljust(widths[c]) for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    lines = [header, sep]
    for row in rows:
        lines.append(" | ".join(str(row[c]).ljust(widths[c]) for c in cols))
    return "\n".join(lines)


def median_abs(y: np.ndarray) -> float:
    return float(np.median(np.abs(np.asarray(y, dtype=float))))


In [ ]:
# Evaluate Ridge across feature sets for each (n, target).
results: dict[float, dict[str, dict[str, dict[str, object]]]] = {}

for n_val in n_classes:
    class_mask = np.isclose(n_all, n_val)
    a_class = a_all[class_mask]
    r_class = r_all[class_mask]
    results[n_val] = {}
    for target_name, y_full in targets.items():
        y = y_full[class_mask]
        results[n_val][target_name] = evaluate_ridge_feature_sets(
            a_values=a_class,
            aspect_ratio_values=r_class,
            y_values=y,
            feature_sets=PHYSICAL_FEATURE_SET_NAMES,
        )

print("Evaluation complete.")


In [ ]:
# Compact LOAO/LOARO summary for all feature sets (main + diagnostic targets).
rows_protocol_compact: list[dict[str, object]] = []
for protocol in protocols:
    for n_val in n_classes:
        for target_name in [*main_targets, *diagnostic_targets]:
            for feature_set in PHYSICAL_FEATURE_SET_NAMES:
                overall = results[n_val][target_name][feature_set][protocol]["overall"]
                rows_protocol_compact.append(
                    {
                        "protocol": protocol,
                        "n": f"{n_val:.1f}",
                        "target": target_name,
                        "feature_set": feature_set,
                        "MAE": compact(float(overall["mae"])),
                        "RMSE": compact(float(overall["rmse"])),
                        "MaxAE": compact(float(overall["max_abs_error"])),
                    }
                )

compact_cols = ["protocol", "n", "target", "feature_set", "MAE", "RMSE", "MaxAE"]
print("Compact LOAO/LOARO comparison: Set A vs B vs C (Ridge-only)")
print(format_table(rows_protocol_compact, compact_cols))


In [ ]:
# Winner matrices by lowest MAE for main targets.
def build_winner_matrix_rows(protocol: str) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for n_val in n_classes:
        row: dict[str, object] = {"n": f"{n_val:.1f}"}
        for target_name in main_targets:
            best_set, _ = select_best_feature_set_by_mae(
                results[n_val][target_name],
                protocol=protocol,
            )
            row[target_name] = best_set
        rows.append(row)
    return rows


winner_cols = ["n", *main_targets]
winner_loao = build_winner_matrix_rows("LOAO")
winner_loaro = build_winner_matrix_rows("LOARO")

print("Winner matrix (best feature set by MAE) - LOAO")
print(format_table(winner_loao, winner_cols))
print("\nWinner matrix (best feature set by MAE) - LOARO")
print(format_table(winner_loaro, winner_cols))


In [ ]:
# Final compact best-only table for each (n, target, protocol).
final_best_rows: list[dict[str, object]] = []

for protocol in protocols:
    for n_val in n_classes:
        class_mask = np.isclose(n_all, n_val)
        for target_name in main_targets:
            best_set, best_overall = select_best_feature_set_by_mae(
                results[n_val][target_name],
                protocol=protocol,
            )
            scale = median_abs(targets[target_name][class_mask])
            ratio = float(best_overall["max_abs_error"] / scale) if scale > 0.0 else np.nan
            final_best_rows.append(
                {
                    "protocol": protocol,
                    "n": f"{n_val:.1f}",
                    "target": target_name,
                    "best_feature_set": best_set,
                    "MAE": compact(best_overall["mae"]),
                    "RMSE": compact(best_overall["rmse"]),
                    "MaxAE": compact(best_overall["max_abs_error"]),
                    "median|y|": compact(scale),
                    "MaxAE/median|y|": compact(ratio),
                }
            )

final_cols = [
    "protocol",
    "n",
    "target",
    "best_feature_set",
    "MAE",
    "RMSE",
    "MaxAE",
    "median|y|",
    "MaxAE/median|y|",
]

print("Final compact baseline table (best feature set per n-target-protocol)")
print(format_table(final_best_rows, final_cols))


In [ ]:
# dE2 diagnostic-only report (kept visible but not headline criterion).
dE2_diag_rows: list[dict[str, object]] = []
eps_dE2 = 5e-3

for protocol in protocols:
    for n_val in n_classes:
        class_mask = np.isclose(n_all, n_val)
        y = targets["dE2"][class_mask]
        scale = median_abs(y)
        near_zero_frac = float(np.mean(np.abs(y) < eps_dE2))
        best_set, best_overall = select_best_feature_set_by_mae(
            results[n_val]["dE2"],
            protocol=protocol,
        )
        ratio = float(best_overall["max_abs_error"] / scale) if scale > 0.0 else np.nan
        dE2_diag_rows.append(
            {
                "protocol": protocol,
                "n": f"{n_val:.1f}",
                "best_feature_set": best_set,
                "MAE": compact(best_overall["mae"]),
                "RMSE": compact(best_overall["rmse"]),
                "MaxAE": compact(best_overall["max_abs_error"]),
                "median|dE2|": compact(scale),
                "MaxAE/median|dE2|": compact(ratio),
                "frac(|dE2|<eps)": f"{near_zero_frac:.3f}",
            }
        )

diag_cols = [
    "protocol",
    "n",
    "best_feature_set",
    "MAE",
    "RMSE",
    "MaxAE",
    "median|dE2|",
    "MaxAE/median|dE2|",
    "frac(|dE2|<eps)",
]

print("dE2 diagnostic report (not a headline success criterion for this linear baseline)")
print(format_table(dE2_diag_rows, diag_cols))
print("\nInterpretation note: dE2 is retained for transparency and stress diagnostics only.")


## Consolidation interpretation checklist

- For `E0`: check whether Set C is consistently strongest across `n` and protocol.
- For `dE1` and `dE3`: check whether Set B or Set C is dominant.
- Keep claims bounded to this validated fixed discrete-`n` window.
- Treat this notebook as the final **Ridge baseline consolidation** artifact for the chapter.